In [3]:
#Install Dependencies
%pip install --upgrade --quiet \
    google-cloud-bigquery \
    google-cloud-bigquery-connection \
    "google-genai>=1.51.0" \
    db-dtypes "pandas==2.2.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 80.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.54.0 which is incompatible.


In [5]:
# Configuration and enable APIs

PROJECT_ID   = "qwiklabs-gcp-02-a9a98f0a78c1"
BQ_LOCATION  = "us-central1"             # single region for dataset + connection + model
DATASET_ID   = "aurora_bay"
CONN_ID      = "bqml_vertex_conn"        # Cloud-resource connection name

# Source data (provided by the challenge)
GCS_URI      = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"

# Fully-qualified table / model names
FAQ_TABLE     = f"{PROJECT_ID}.{DATASET_ID}.aurora_bay_faqs"
FAQ_EMB_TABLE = f"{PROJECT_ID}.{DATASET_ID}.aurora_bay_faqs_embedded"
EMBED_MODEL   = f"{PROJECT_ID}.{DATASET_ID}.embedding_model"
TEXT_MODEL    = f"{PROJECT_ID}.{DATASET_ID}.text_model"   # used only by the Section 10 SQL variant

# Embedding model endpoint (current BigQuery default text embedding model)
EMBED_ENDPOINT = "text-embedding-005"

# Gemini settings for the final answer step (google-genai SDK).
GEMINI_LOCATION = "global"             # works for the 2.5 family and is required for 3.x
MODEL_ID        = "gemini-2.5-flash"   # broadly available in labs; swap if your project has more

# Enable the APIs needed for BigQuery + remote Vertex models (safe to re-run).
!gcloud services enable bigquery.googleapis.com bigqueryconnection.googleapis.com \
    aiplatform.googleapis.com --project={PROJECT_ID}

Operation "operations/acat.p2-737059363010-729fb043-54c5-47c3-b372-50ce60a707ee" finished successfully.


In [15]:
# Load the FAQ CSV into BigQuery
load_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,       # skip header row
    autodetect=True,           # infer column names + types
    write_disposition="WRITE_TRUNCATE",
)

load_job = bq.load_table_from_uri(GCS_URI, FAQ_TABLE, job_config=load_config)
load_job.result()              # wait for completion

tbl = bq.get_table(FAQ_TABLE)
print(f"Loaded {tbl.num_rows} rows into {FAQ_TABLE}\n")
print("Schema:")
for f in tbl.schema:
    print(f"  - {f.name} ({f.field_type})")

print("\nPreview:")
bq.query(f"SELECT * FROM `{FAQ_TABLE}` LIMIT 5").to_dataframe()

Loaded 50 rows into qwiklabs-gcp-02-a9a98f0a78c1.aurora_bay.aurora_bay_faqs

Schema:
  - string_field_0 (STRING)
  - string_field_1 (STRING)

Preview:


,string_field_0,string_field_1
0,When was Aurora Bay founded?,Aurora Bay was founded in 1901 by a group of f...
1,What is the population of Aurora Bay?,Aurora Bay has a population of approximately 3...
2,Where is the Aurora Bay Town Hall located?,The Town Hall is located at 100 Harbor View Ro...
3,Who is the current mayor of Aurora Bay?,"The current mayor is Linda Greenwood, elected ..."
4,What are the primary industries in Aurora Bay?,The primary industries include commercial fish...


In [6]:
# Create the BigQuery dataset

from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
ds.location = BQ_LOCATION
bq.create_dataset(ds, exists_ok=True)
print(f"Dataset ready: {PROJECT_ID}.{DATASET_ID} ({BQ_LOCATION})")

Dataset ready: qwiklabs-gcp-02-a9a98f0a78c1.aurora_bay (us-central1)


In [8]:
# Load FAQ CSV into BigQuery

load_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,       # skip header row
    autodetect=True,           # infer column names + types
    write_disposition="WRITE_TRUNCATE",
)

load_job = bq.load_table_from_uri(GCS_URI, FAQ_TABLE, job_config=load_config)
load_job.result()              # wait for completion

tbl = bq.get_table(FAQ_TABLE)
print(f"Loaded {tbl.num_rows} rows into {FAQ_TABLE}\n")
print("Schema:")
for f in tbl.schema:
    print(f"  - {f.name} ({f.field_type})")

print("\nPreview:")
bq.query(f"SELECT * FROM `{FAQ_TABLE}` LIMIT 5").to_dataframe()

Loaded 50 rows into qwiklabs-gcp-02-a9a98f0a78c1.aurora_bay.aurora_bay_faqs

Schema:
  - string_field_0 (STRING)
  - string_field_1 (STRING)

Preview:


,string_field_0,string_field_1
0,When was Aurora Bay founded?,Aurora Bay was founded in 1901 by a group of f...
1,What is the population of Aurora Bay?,Aurora Bay has a population of approximately 3...
2,Where is the Aurora Bay Town Hall located?,The Town Hall is located at 100 Harbor View Ro...
3,Who is the current mayor of Aurora Bay?,"The current mayor is Linda Greenwood, elected ..."
4,What are the primary industries in Aurora Bay?,The primary industries include commercial fish...


In [21]:
# Detect the question and answer columns

cols = [f.name for f in tbl.schema]

def _pick(candidates, default):
    for c in cols:
        if c.lower() in candidates:
            return c
    return default

QUESTION_COL = _pick({"question", "questions", "q"}, cols[0])
ANSWER_COL   = _pick({"answer", "answers", "a", "response"}, cols[-1])

# SQL expression for the text to embed (NULL-safe).
CONTENT_EXPR = (
    f"CONCAT('Question: ', IFNULL(CAST({QUESTION_COL} AS STRING), ''), "
    f"'\nAnswer: ', IFNULL(CAST({ANSWER_COL} AS STRING), ''))"
)

print(f"Question column: {QUESTION_COL}")
print(f"Answer column:   {ANSWER_COL}")
print(f"Embedding input: {CONTENT_EXPR}")
# If auto-detection picked the wrong columns, set QUESTION_COL / ANSWER_COL by hand here.

CONTENT_EXPR = (
    f"CONCAT('Question: ', IFNULL(CAST({QUESTION_COL} AS STRING), ''), "
    f"' | Answer: ', IFNULL(CAST({ANSWER_COL} AS STRING), ''))"
)

Question column: string_field_0
Answer column:   string_field_1
Embedding input: CONCAT('Question: ', IFNULL(CAST(string_field_0 AS STRING), ''), '
Answer: ', IFNULL(CAST(string_field_1 AS STRING), ''))


In [18]:
import time, subprocess
from google.cloud import bigquery_connection_v1 as bqconn
from google.api_core.exceptions import NotFound

conn_client = bqconn.ConnectionServiceClient()
parent      = f"projects/{PROJECT_ID}/locations/{BQ_LOCATION}"
conn_name    = f"{parent}/connections/{CONN_ID}"

try:
    conn = conn_client.get_connection(name=conn_name)
    print(f"Using existing connection: {CONN_ID}")
except NotFound:
    conn = conn_client.create_connection(
        parent=parent,
        connection_id=CONN_ID,
        connection=bqconn.Connection(cloud_resource=bqconn.CloudResourceProperties()),
    )
    print(f"Created connection: {CONN_ID}")

CONN_SA = conn.cloud_resource.service_account_id
print(f"Connection service account: {CONN_SA}")

def grant_vertex_role(max_attempts=8, delay=15):
    """Grant roles/aiplatform.user, retrying until the new SA is visible to IAM."""
    cmd = ["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
           f"--member=serviceAccount:{CONN_SA}",
           "--role=roles/aiplatform.user", "--condition=None", "--quiet"]
    for attempt in range(1, max_attempts + 1):
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode == 0:
            print(f"Granted roles/aiplatform.user (attempt {attempt}).")
            return True
        if "does not exist" in res.stderr:
            print(f"Attempt {attempt}: SA not visible to IAM yet, retrying in {delay}s...")
            time.sleep(delay)
        else:
            print("Unexpected error:\n", res.stderr)
            time.sleep(delay)
    print("IAM grant did NOT succeed after retries - see errors above.")
    return False

if grant_vertex_role():
    print("Waiting 30s for the binding to propagate...")
    time.sleep(30)
    print("Done.")

Using existing connection: bqml_vertex_conn
Connection service account: bqcx-737059363010-kd6g@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Granted roles/aiplatform.user (attempt 1).
Waiting 30s for the binding to propagate...
Done.


In [19]:
# Create the remote embedding model
create_embed_model = f"""
CREATE OR REPLACE MODEL `{EMBED_MODEL}`
REMOTE WITH CONNECTION `{PROJECT_ID}.{BQ_LOCATION}.{CONN_ID}`
OPTIONS (ENDPOINT = '{EMBED_ENDPOINT}')
"""
bq.query(create_embed_model).result()
print(f"Created remote embedding model: {EMBED_MODEL}")

Created remote embedding model: qwiklabs-gcp-02-a9a98f0a78c1.aurora_bay.embedding_model


In [22]:
# generate and store embeddings
create_embeddings = f"""
CREATE OR REPLACE TABLE `{FAQ_EMB_TABLE}` AS
SELECT *
FROM AI.GENERATE_EMBEDDING(
  MODEL `{EMBED_MODEL}`,
  (SELECT *, {CONTENT_EXPR} AS content FROM `{FAQ_TABLE}`)
)
"""
bq.query(create_embeddings).result()

# Verify: any row with a non-empty status failed to embed.
check = bq.query(f"""
  SELECT COUNT(*) AS total,
         COUNTIF(LENGTH(status) > 0) AS errors
  FROM `{FAQ_EMB_TABLE}`
""").to_dataframe()
print(check)
print(f"\nEmbeddings stored in {FAQ_EMB_TABLE}")
if int(check.errors[0]) > 0:
    print("Some rows failed (likely transient Vertex quota) - re-run this cell to retry.")

   total  errors
0     50       0

Embeddings stored in qwiklabs-gcp-02-a9a98f0a78c1.aurora_bay.aurora_bay_faqs_embedded


In [23]:
create_embed_model = f"""
CREATE OR REPLACE MODEL `{EMBED_MODEL}`
REMOTE WITH CONNECTION `{PROJECT_ID}.{BQ_LOCATION}.{CONN_ID}`
OPTIONS (ENDPOINT = '{EMBED_ENDPOINT}')
"""
bq.query(create_embed_model).result()
print(f"Created remote embedding model: {EMBED_MODEL}")

Created remote embedding model: qwiklabs-gcp-02-a9a98f0a78c1.aurora_bay.embedding_model


In [24]:
# Vector Search
def vector_search(question: str, k: int = 5):
    """Return the top-k most relevant FAQ rows for a question, as a DataFrame."""
    sql = f"""
    SELECT base.{QUESTION_COL} AS question,
           base.{ANSWER_COL}   AS answer,
           distance
    FROM VECTOR_SEARCH(
      TABLE `{FAQ_EMB_TABLE}`, 'embedding',
      (
        SELECT embedding
        FROM AI.GENERATE_EMBEDDING(
          MODEL `{EMBED_MODEL}`,
          (SELECT @q AS content)
        )
      ),
      top_k => {int(k)},
      distance_type => 'COSINE'
    )
    ORDER BY distance
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("q", "STRING", question)]
    )
    return bq.query(sql, job_config=job_config).to_dataframe()

# Smoke test
vector_search("What should I do in an emergency?", k=3)

,question,answer,distance
0,How can I find local emergency shelters during...,Emergency shelters are typically set up at Aur...,0.486190
1,How can local residents receive alerts about e...,Residents can sign up for text or email alerts...,0.490481
2,How do I contact the Aurora Bay Fire Department?,The volunteer-based Aurora Bay Fire Department...,0.521153


In [25]:
# The grounded chatbot
from google import genai
from google.genai import types

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=GEMINI_LOCATION)

SYSTEM_INSTRUCTION = """\
You are the information assistant for the town of Aurora Bay, Alaska.
Answer the user's question using ONLY the FAQ context provided in the prompt.
- If the context contains the answer, respond concisely and helpfully.
- If the context does NOT contain the answer, say you do not have that information
  in the Aurora Bay FAQs and suggest contacting the town office - do not guess.
Never fabricate facts that are not supported by the provided context.
"""

def answer_question(question: str, k: int = 5, show_context: bool = True) -> str:
    df = vector_search(question, k)
    if df.empty:
        return "I couldn't find anything relevant in the Aurora Bay FAQs."

    context = "\n\n".join(
        f"Q: {row.question}\nA: {row.answer}" for row in df.itertuples()
    )

    if show_context:
        print("Retrieved FAQs (by cosine distance):")
        for row in df.itertuples():
            print(f"  - ({row.distance:.4f}) {row.question}")
        print()

    prompt = f"FAQ context:\n{context}\n\nUser question: {question}\n\nAnswer:"
    resp = genai_client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0.2,
            max_output_tokens=1024,
        ),
    )
    return (resp.text or "").strip()

print("RAG chatbot ready.")

RAG chatbot ready.


In [26]:
# Demonstration
questions = [
    "How do I report a power outage?",
    "What are the library's hours?",
    "Is there a leash law for dogs?",
    "When is the farmers market held?",
]

for q in questions:
    print("=" * 78)
    print(f"USER: {q}")
    print("-" * 78)
    print(answer_question(q, k=4))
    print()

USER: How do I report a power outage?
------------------------------------------------------------------------------
Retrieved FAQs (by cosine distance):
  - (0.1955) How do I report a power outage?
  - (0.4520) How can local residents receive alerts about emergencies or important updates?
  - (0.4842) How do I request a building permit?
  - (0.4856) How can I find local emergency shelters during severe weather?

Report power outages to the Aurora Bay Utilities Department at (907) 555-0101 or submit an online form on the town’s utility portal.

USER: What are the library's hours?
------------------------------------------------------------------------------
Retrieved FAQs (by cosine distance):
  - (0.2911) What are the operating hours of the Aurora Bay Public Library?
  - (0.4354) When are the town council meetings held?
  - (0.4359) Does Aurora Bay have a public library?
  - (0.4487) Are there specific quiet hours or noise ordinances?

The Aurora Bay Public Library is open Monday thro